# LAB-08 메일링 리스트 / 데이터 베이스 연동 
<학습 내용>
- 메일 발송 모듈 제작
- 메일링 리스트 구현
- 비동기 처리
- 비동기 리턴/ 예외 처리
- 데이터 입력/수정/삭제

### 메일 발생 모듈 제작

In [ ]:
# -> 경로 정보를 취득하기 위한 모듈
import os
# -> 발송서버와 연동하기 위한 모듈
from smtplib import SMTP
# -> 본문 구성 기능
from email.mime.text import MIMEText
# -> 파일을 Multipart 형식으로 변환
from email.mime.application import MIMEApplication
# -> 파일을 본문에 추가하는 기능 제공
from email.mime.multipart import MIMEMultipart

In [15]:
def sendMail(from_addr, to_addr, subject, content, files=[]): 
  # 컨텐츠 형식 (plain or html)
  content_type ='plain'
  # 로그인 계정 이름 (네이버=아이디, 구글=메일주소)
  username = "bohee0506@gmail.com"
  # 비밀번호 (네이버=개인비밀번호,애플리케이션 비밀번호, 구글=앱 비밀번호
  password ="xtth ykdj asna jjbb"

  # 구글 발송 서버 주소와 포트 (고정값)
  smtp ="smtp.gmail.com"
  port = 587

  # 메일 발송 정보를 저장하기 위한 객체
  msg = MIMEMultipart()

  msg['Subject'] =subject # 메일 제목
  msg['From'] =from_addr  # 보내는 사람
  msg['To'] =to_addr # 받는 사람

  # 본문 설정 -> 메일의 내용과 형식 지정
  msg.attach(MIMEText(content,content_type))


  # 리스트 변수의 원소가 하나라도 존재할 경우 True
  if files:
    for file_item in files:
      if os.path.exists(file_item):
        # 바이너리(b) 형식으로 읽기(r)
        with open(file_item,'rb') as f:

          # 전체 경로에서 파일의 이름만 추출
          basename =os.path.basename(file_item)
          # 파일의 내용과 파일이름을 메일에 첨부할 형식으로 변환
          part =MIMEApplication(f.read(),Name = basename)
          # 파일첨부
          part['Contenr-Dispositin'] = 'attachment; filename="%s"' %basename
          msg.attach(part)

          print(basename,"(이)가 첨부되었습니다")



  mail = SMTP (smtp)
  #메일 서버 접속
  mail.ehlo()
  #메일 서버 연동 설정
  mail.starttls()
  #메일 서버 로그인
  mail.login(username,password)
  #메일 보내기
  mail.sendmail(from_addr,to_addr,msg.as_string())
  #메일 서버 접속 종료
  mail.quit()


if __name__ == "__main__":
  sendMail ("bohee0506@gmail.com","bohee0506@gmail.com","메일 발생 모듈 테스트","test 입니다" ,files =["hello.txt"])



hello.txt (이)가 첨부되었습니다


### 메일링 리스트 구현

In [ ]:
from mylibrary import MyMailer
import datetime as dt


#날짜 성분값 초기화
today =dt.datetime.now()
year = today.year
month =today.month
day = today.day
print(year,month,day)

2025 10 27


In [ ]:
#보내는 사람, 메일 제목
fromAddr ="운영지원팀 <bohee0506@gmail.com>"

#메일 제목을 위한 템플릿
subjectTmpl ="{name} 님의 {yy} {mm} 월 급여명세서 입니다"

In [ ]:
#메일 본문 가져오기
with open('mail/content.txt','r',encoding='utf-8') as f:
  contentTmpl =f.read()
  print(contentTmpl)

안녕하세요 {name}님

{yy}년도 {mm}월 급여명세서와 결산보고서 보내드립니다.

귀하의 노고에 감사드립니다.

- {yy}년 {mm}월 {dd}일 / 운영지원팀 드림


In [ ]:
#수신자 목록 csv 에 대한 반복 처리
with open("mail/mail_list.csv","r",encoding="euc-kr") as f:
  csv =f.readlines()


  for line in csv:
    name,email,file1,file2 = line.strip().split(",")
    #print(name,email,file1,file2)

    toAddr ="{name} <{email}>".format(name=name , email=email)
    #print(toAddr)

    #메일 제목
    subject = subjectTmpl.format(name=name,yy=year, mm=month)

    #메일 내용
    content = contentTmpl.format(name=name,yy=year,mm=month,dd=day)

    #메일 보내기

    #보낼 떄 아래 함수 주석 풀기
    #MyMailer.sendMail (fromAddr,toAddr,subject,content,[file1,file2])

document.pptx (이)가 첨부되었습니다
pay1.xlsx (이)가 첨부되었습니다
document.pptx (이)가 첨부되었습니다
pay2.xlsx (이)가 첨부되었습니다
pay3.xlsx (이)가 첨부되었습니다


## 비동기처리

동기처리는 직렬 , 즉 순차적으로 작업을 수행하면서 전체 실행 시간이 느림
비동기 처리는 병렬, 즉 동시 작업이므로 전체 실행 시간이 빠름


딜레이를 거는 모듈은 time 

동시에 작업 가능한 갯수 (mas_worker)

with 구문은 작업이 다 끝나는걸 기다림

In [ ]:
#프로그램에 딜레이를 적용하는 기능을 적용하는 모듈
import time

#날짜를 처리하는 모듈
import datetime as dt
#앞서 구현한 메일 발생 모듈
from mylibrary import MyMailer
#비동기 처리 기능을 제공하는 모듈
import concurrent.futures as futures

In [24]:
#날짜 성분값 초기화
today =dt.datetime.now()
year = today.year
month =today.month
day = today.day

#보내는 사람, 메일 제목
fromAddr ="운영지원팀 <bohee0506@gmail.com>"

#메일 제목을 위한 템플릿
subjectTmpl ="{name} 님의 {yy} {mm} 월 급여명세서 입니다"

#메일 본문 가져오기
with open('mail/content.txt','r',encoding='utf-8') as f:
  contentTmpl =f.read()
  print(contentTmpl)

안녕하세요 {name}님

{yy}년도 {mm}월 급여명세서와 결산보고서 보내드립니다.

귀하의 노고에 감사드립니다.

- {yy}년 {mm}월 {dd}일 / 운영지원팀 드림


## 동기식 메일 발송
순차적으로 메일이 발송되며, 하나의 발송이 종료되어야만 다음 메일을 발송한다

In [ ]:
startTime = dt.datetime.now()

#수신자 목록 csv 에 대한 반복 처리
with open("mail/mail_list.csv","r",encoding="euc-kr") as f:
  csv =f.readlines()


  for line in csv:
    name,email,file1,file2 = line.strip().split(",")
    #print(name,email,file1,file2)

    toAddr ="{name} <{email}>".format(name=name , email=email)
    #print(toAddr)

    #메일 제목
    subject = subjectTmpl.format(name=name,yy=year, mm=month)

    #메일 내용
    content = contentTmpl.format(name=name,yy=year,mm=month,dd=day)

    #메일 보내기

    #보낼 떄 아래 함수 주석 풀기
    #MyMailer.sendMail (fromAddr,toAddr,subject,content,[file1,file2])


endTime =dt.datetime.now()
workTime =endTime -startTime
print("작업에 소요된 시간은 총 %s 초 입니다" %workTime.seconds)

document.pptx (이)가 첨부되었습니다
pay1.xlsx (이)가 첨부되었습니다
document.pptx (이)가 첨부되었습니다
pay2.xlsx (이)가 첨부되었습니다
pay3.xlsx (이)가 첨부되었습니다
작업에 소요된 시간은 총 12 초 입니다


## 비동기식 메일 발송
max_workers 로 설정된 값에 따라 최대 10개까지 동시 발송을 수행하며, 총 작업 시간이 단축된 것을 확인할 수 있다

In [28]:
startTime = dt.datetime.now()

#수신자 목록 csv 에 대한 반복 처리
with open("mail/mail_list.csv","r",encoding="euc-kr") as f:
  csv =f.readlines()

  #비동기식 처리를 위한 with 구문 추가
  with futures.ThreadPoolExecutor(max_workers=10) as executor:
    
    for line in csv:
      name,email,file1,file2 = line.strip().split(",")
      #print(name,email,file1,file2)
      toAddr ="{name} <{email}>".format(name=name , email=email)
      #print(toAddr)
      #메일 제목
      subject = subjectTmpl.format(name=name,yy=year, mm=month)
      #메일 내용
      content = contentTmpl.format(name=name,yy=year,mm=month,dd=day)

      #동기식 실행을 위한 구문 / 
      #executor.submit(MyMailer.sendMail,fromAddr,toAddr,subject,content,[file1,file2])


endTime =dt.datetime.now()
workTime =endTime -startTime
print("작업에 소요된 시간은 총 %s 초 입니다" %workTime.seconds)

작업에 소요된 시간은 총 0 초 입니다


## 비동리 리턴/예외 처리

In [32]:
#비동기 작업을 위한 함수 정의

import random

def randomWork(name):
  randomSecond = random.randrange(1,9)
  print("[%s] 작업을 %d 초 동안 수행합니다" %(name,randomSecond))


  for i in range(0,randomSecond):
    time.sleep(1)
    print("[%s] %d초 ..."%(name,i+1))

  
  print("[%s] 작업이 종료되었습니다" %name)

  return randomSecond


# ====================================================
#비동기 작업 처리 후 결과값 리턴 받기
startTime = dt.datetime.now()


names=['Lee','Kim','Hong','Park','Nam']
processes =[]
resultSet=[]


with futures.ThreadPoolExecutor(max_workers=3) as executor:
  for n in names:
    pro = executor.submit(randomWork,n)
    processes.append(pro)



  for p in processes:
    result=p.result()
    resultSet.append(result)



  print("비동기 처리가 총 %d 초 의 작업을 수행했습니다" %sum(resultSet))


  endTime =dt.datetime.now()
  workTime =endTime -startTime
  print("작업에 소요된 시간은 총 %s 초 입니다" % workTime.seconds)

[Lee] 작업을 7 초 동안 수행합니다
[Kim] 작업을 5 초 동안 수행합니다
[Hong] 작업을 3 초 동안 수행합니다
[Hong] 1초 ...[Kim] 1초 ...
[Lee] 1초 ...

[Hong] 2초 ...[Lee] 2초 ...

[Kim] 2초 ...
[Hong] 3초 ...[Lee] 3초 ...

[Hong] 작업이 종료되었습니다
[Park] 작업을 1 초 동안 수행합니다
[Kim] 3초 ...
[Lee] 4초 ...
[Park] 1초 ...
[Park] 작업이 종료되었습니다
[Nam] 작업을 7 초 동안 수행합니다
[Kim] 4초 ...
[Lee] 5초 ...[Nam] 1초 ...

[Kim] 5초 ...
[Kim] 작업이 종료되었습니다
[Nam] 2초 ...
[Lee] 6초 ...
[Nam] 3초 ...[Lee] 7초 ...
[Lee] 작업이 종료되었습니다

[Nam] 4초 ...
[Nam] 5초 ...
[Nam] 6초 ...
[Nam] 7초 ...
[Nam] 작업이 종료되었습니다
비동기 처리가 총 23 초 의 작업을 수행했습니다
작업에 소요된 시간은 총 11 초 입니다


In [35]:
#예외를 발생시키는 함수 정의
import random

seasons = ["봄","여름","가을","겨울"]

def seasonWork(name):
  randomSecond = random.randrange(1,9)

  print("[%s] 작업을 %d 초 동안 수행합니다" % (name,randomSecond))

  for i in range (0,randomSecond):
    time.sleep(1)
    print("[%s] 지금은 <%s> 입니다" %(name,seasons[i]))


  print("[%s] 작업이 종료되었습니다" %name)

  return [name,randomSecond]


startTime = dt.datetime.now()

names =['Lee','Kim','Hong','Park','Nam']
processes =[]
nameSet =[]
timeSet =[]


with futures.ThreadPoolExecutor(max_workers=len(names)) as executor:
  for n in names:
    pro=executor.submit(seasonWork,n)
    processes.append(pro)



    for p in processes:
      try :
        name,n=p.result()
        nameSet.append(name)
        timeSet.append(n)
        print("%s의 작업이 완료되었습니다" %name)

      except Exception as e:
        print(e)



endTime = dt.datetime.now()
workTime=endTime-startTime
print("작업에 소요된 시간은 총 %s 초 입니다" % workTime.seconds)

[Lee] 작업을 7 초 동안 수행합니다
[Lee] 지금은 <봄> 입니다
[Lee] 지금은 <여름> 입니다
[Lee] 지금은 <가을> 입니다
[Lee] 지금은 <겨울> 입니다
list index out of range
list index out of range
[Kim] 작업을 6 초 동안 수행합니다
[Kim] 지금은 <봄> 입니다
[Kim] 지금은 <여름> 입니다
[Kim] 지금은 <가을> 입니다
[Kim] 지금은 <겨울> 입니다
list index out of range
list index out of range
list index out of range
[Hong] 작업을 1 초 동안 수행합니다
[Hong] 지금은 <봄> 입니다
[Hong] 작업이 종료되었습니다
Hong의 작업이 완료되었습니다
list index out of range
list index out of range
Hong의 작업이 완료되었습니다
[Park] 작업을 2 초 동안 수행합니다
[Park] 지금은 <봄> 입니다
[Park] 지금은 <여름> 입니다
[Park] 작업이 종료되었습니다
Park의 작업이 완료되었습니다
list index out of range
list index out of range
Hong의 작업이 완료되었습니다
Park의 작업이 완료되었습니다
[Nam] 작업을 1 초 동안 수행합니다
[Nam] 지금은 <봄> 입니다
[Nam] 작업이 종료되었습니다
Nam의 작업이 완료되었습니다
작업에 소요된 시간은 총 14 초 입니다
